---
## PARTIE 1 — Chargement et préparation des données

### Étape 1 — Chargement du dataset depuis S3

In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, DoubleType

spark = SparkSession.builder.getOrCreate()

S3_PATH = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
df_raw = spark.read.json(S3_PATH)

print(f"Dataset chargé : {df_raw.count():,} lignes")

Dataset chargé : 55,691 lignes


### Étape 2 — Inspection du schéma

In [0]:
df_raw.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

**Observations :**
- Toutes les données sont dans une colonne racine "data" qui contient des champs imbriqués, il faut l'aplatir.

### Étape 3 — Aplatir la colonne racine "data"

In [0]:
df = df_raw.select(
    F.col("data.appid"),
    F.col("data.name"),
    F.col("data.developer"),
    F.col("data.publisher"),
    F.col("data.genre"),
    F.col("data.release_date"),
    F.col("data.required_age"),
    F.col("data.price"),
    F.col("data.initialprice"),
    F.col("data.discount"),
    F.col("data.positive"),
    F.col("data.negative"),
    F.col("data.languages"),
    F.col("data.platforms"),)

print(f"{df.count():,} jeux chargés, {len(df.columns)} colonnes")
df.show(3, truncate=True)

55,691 jeux chargés, 14 colonnes
+-------+--------------+--------------------+--------------------+--------------------+------------+------------+-----+------------+--------+--------+--------+--------------------+--------------------+
|  appid|          name|           developer|           publisher|               genre|release_date|required_age|price|initialprice|discount|positive|negative|           languages|           platforms|
+-------+--------------+--------------------+--------------------+--------------------+------------+------------+-----+------------+--------+--------+--------+--------------------+--------------------+
|     10|Counter-Strike|               Valve|               Valve|              Action|   2000/11/1|           0|  999|         999|       0|  201215|    5199|English, French, ...|  {true, true, true}|
|1000000|     ASCENXION|IndigoBlue Game S...|PsychoFlux Entert...|Action, Adventure...|  2021/05/14|           0|  999|         999|       0|      27|       5|

**Observations :**
- On a maintenant un DataFrame plat avec les colonnes essentielles à l'analyse

### Étape 4 — Nettoyage des types et création de colonnes calculées


In [0]:
df_clean = df \
    .withColumn("price", F.col("price").cast(DoubleType()) / 100) \
    .withColumn("initialprice", F.col("initialprice").cast(DoubleType()) / 100) \
    .withColumn("discount", F.col("discount").cast(DoubleType())) \
    .withColumn("required_age", F.expr("try_cast(required_age as int)")) \
    .withColumn("release_date",
        F.coalesce(
            F.expr("try_to_date(release_date, 'yyyy/MM/d')"),
            F.expr("try_to_date(release_date, 'yyyy/MM')"),
            F.expr("try_to_date(release_date, 'yyyy')")
        )
    ) \
    .withColumn("release_year",   F.year(F.col("release_date"))) \
    .withColumn("total_reviews",  F.col("positive") + F.col("negative")) \
    .withColumn("positive_ratio",
        F.when(F.col("total_reviews") > 0,
            F.round(F.col("positive") / F.col("total_reviews") * 100, 2)
        ).otherwise(None)
    ) \
    .withColumn("on_windows", F.col("platforms").getField("windows")) \
    .withColumn("on_mac", F.col("platforms").getField("mac")) \
    .withColumn("on_linux",F.col("platforms").getField("linux")) \
    .drop("platforms")

df_clean.select("name", "price", "release_year", "positive_ratio").show(5, truncate=False)

+---------------------------+-----+------------+--------------+
|name                       |price|release_year|positive_ratio|
+---------------------------+-----+------------+--------------+
|Counter-Strike             |9.99 |2000        |97.48         |
|ASCENXION                  |9.99 |2021        |84.38         |
|Crown Trick                |5.99 |2020        |86.19         |
|Cook, Serve, Delicious! 3?!|19.99|2020        |93.2          |
|细胞战争                   |1.99 |2019        |0.0           |
+---------------------------+-----+------------+--------------+
only showing top 5 rows


**Observations :**
- Les prix sont maintenant en euros, les dates exploitables, et les plateformes accessibles directement
- Le DataFrame "df_clean" sera notre base pour la majorité des analyses

### Étape 5 — Création de 2 DataFrames spécialisés (langues et genres)

In [0]:
# DataFrame des langues : 1 ligne par couple (jeu, langue)
df_languages = df_clean \
    .withColumn("language", F.explode(F.split(F.col("languages"), ",\\s*"))) \
    .withColumn("language", F.trim(F.col("language"))) \
    .select("appid", "name", "language") \
    .filter(F.col("language").isNotNull() & (F.col("language") != ""))

# DataFrame des genres : 1 ligne par couple (jeu, genre)
# On garde toutes les colonnes utiles (publisher, price, ratings, plateformes...)
# pour pouvoir faire des analyses croisées sans devoir re-joindre avec df_clean
df_genres = df_clean \
    .withColumn("genre_clean", F.explode(F.split(F.col("genre"), ",\\s*"))) \
    .withColumn("genre_clean", F.trim(F.col("genre_clean"))) \
    .filter(F.col("genre_clean").isNotNull() & (F.col("genre_clean") != ""))

print(f"df_clean     : {df_clean.count():,} lignes (1 par jeu)")
print(f"df_languages : {df_languages.count():,} lignes (1 par langue/jeu)")
print(f"df_genres    : {df_genres.count():,} lignes (1 par genre/jeu)")

df_clean     : 55,691 lignes (1 par jeu)
df_languages : 202,293 lignes (1 par langue/jeu)
df_genres    : 157,110 lignes (1 par genre/jeu)


**Observations :**
- "df_clean" reste notre base principale (1 ligne = 1 jeu).
- "df_languages" permet de compter les jeux par langue.
- "df_genres" permet d'analyser chaque genre indépendamment (un jeu peut avoir plusieurs genres comme "Action, Adventure, RPG").

## PARTIE 2 — Analyse macro

### Étape 6 — Q1 : Quels publishers ont sorti le plus de jeux ?

In [0]:
top_publishers = df_clean \
    .filter(F.col("publisher").isNotNull() & (F.col("publisher") != "")) \
    .groupBy("publisher") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .limit(15)

display(top_publishers)

publisher,count
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
HH-Games,132
Sekai Project,132
Ubisoft,127
Laush Studio,126


Databricks visualization. Run in Databricks to view.

**Observations :**
- Les grands éditeurs AAA (Ubisoft, Square Enix) ne sont pas en haut du classement par volume, ils privilégient peu de jeux mais à fort budget.


### Étape 7 — Q2 : Quels sont les jeux les mieux notés ?

On filtre les jeux avec au moins 100 avis pour éviter les biais. (un jeu avec 1 seul avis positif aurait 100% de ratio mais ne représenterait rien)

In [0]:
best_rated = df_clean \
    .filter(F.col("total_reviews") >= 100) \
    .select("name", "publisher", "positive_ratio", "total_reviews") \
    .orderBy(F.col("positive_ratio").desc(), F.col("total_reviews").desc()) \
    .limit(15)

display(best_rated)

name,publisher,positive_ratio,total_reviews
The Void Rains Upon Her Heart,The Hidden Levels,100.0,496
祈風 Inorikaze,觀象草圖 Astrolabe Draft,100.0,327
秘封旅行 ~ Secret Sealing Travel,鸽屋谷,100.0,218
Elasto Mania Remastered,Elasto Mania Team,100.0,190
Freshly Frosted,The Quantum Astrophysicists Guild,100.0,157
HAYAI,Chaoclypse,100.0,148
FIND ALL 2: Middle Ages,Very Very LITTLE Studio,100.0,132
Touhou Kaeizuka ～ Phantasmagoria of Flower View.,"Mediascape Co., Ltd.",100.0,119
Lucy Dreaming,Tall Story Games Ltd,100.0,118
未来战士,BIFROST CLOUD PTE. LTD.,100.0,116


**Observations :**
- Les jeux les mieux notés ont des ratios proches de 99-100% positifs
- Ce sont souvent des jeux de niche très appréciés par leur communauté.
- Cependant la note d'un jeu reflète son adéquation avec sa cible, pas forcément sa popularité massive

### Étape 8 — Q3 : Sorties par année + impact du Covid

On commence à partir de 2000 et on regarde l'évolution.

In [0]:
releases_per_year = df_clean \
    .filter(F.col("release_year").isNotNull() & (F.col("release_year") >= 2000)) \
    .groupBy("release_year") \
    .count() \
    .orderBy("release_year")

display(releases_per_year)

release_year,count
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61
2007,98
2008,159
2009,311


Databricks visualization. Run in Databricks to view.

**Observations :**
- A partir de 2014 la croissance est explosive, Steam a démocratisé la publication (digitalisation).
- Pic notable en 2020-2021 (Covid) : les studios ont profité du confinement, la demande de divertissement a augmenté, et constate unréduction post-confinemnt.

### Étape 9 — Q4 : Distribution des prix et discounts

On regarde d'abord la distribution des jeux payants (>0€), puis la part de jeux en réduction.

In [0]:
# Distribution des prix (jeux payants entre 0€ et 100€ pour exclure les outliers)
price_dist = df_clean \
    .filter((F.col("price") > 0) & (F.col("price") < 100)) \
    .select("price")

display(price_dist)

price
9.99
9.99
5.99
19.99
1.99
7.99
12.99
2.99
13.99
0.99


Databricks visualization. Run in Databricks to view.

In [0]:
# Part de jeux en réduction
discount_split = df_clean \
    .filter(F.col("discount").isNotNull()) \
    .withColumn("status",
        F.when(F.col("discount") > 0, "En réduction")
         .otherwise("Prix normal")
    ) \
    .groupBy("status") \
    .count()

display(discount_split)

status,count
En réduction,2518
Prix normal,53173


Databricks visualization. Run in Databricks to view.

**Observations :**
- La majorité des jeux payants se situent dans la fourchette 1-10€
- Peu de jeux dépassent les 30€
- Un prix dans la fourchette 1-30€ touche le plus large public.

### Étape 10 — Q5 : Langues les plus représentées

In [0]:
top_languages = df_languages \
    .groupBy("language") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .limit(15)

display(top_languages)

language,count
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6600


Databricks visualization. Run in Databricks to view.

**Observations :**
- Sans surprise l'anglais domine très largement, c'est la langue universelle des jeux vidéo.

### Étape 11 — Q6 : Jeux interdits aux moins de 16 / 18 ans

In [0]:
age_dist = df_clean \
    .withColumn("age_category",
        F.when(F.col("required_age") >= 18, "18+")
         .when(F.col("required_age") >= 16, "16-17")
         .when(F.col("required_age") >= 13, "13-15")
         .otherwise("Tout public")
    ) \
    .groupBy("age_category") \
    .count() \
    .orderBy(F.col("count").desc())

display(age_dist)

age_category,count
Tout public,55086
13-15,300
18+,229
16-17,76


Databricks visualization. Run in Databricks to view.

**Observations :**
- L'immense majorité des jeux Steam sont classés "Tout public" (98.9%)

---
## PARTIE 3 — Analyse par genres

On utilise "df_genres" qui a une ligne par couple (jeu, genre). Un même jeu apparaît donc plusieurs fois s'il a plusieurs genres.

In [0]:
top_genres = df_genres \
    .groupBy("genre_clean") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .limit(15)

display(top_genres)

genre_clean,count
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


Databricks visualization. Run in Databricks to view.

**Observations :**
- Les genres dominants sont "Action", "Indie", "Adventure", "Casual", "Simulation"
- "Indie" est très représenté, Steam est la plateforme de référence des jeux indépendant

### Étape 13 — Q8 : Genres avec le meilleur ratio positif/négatif

On joint "df_genres" avec les notes "positive_ratio" pour calculer le ratio moyen par genre. On filtre sur les jeux avec au moins 100 reviews.

In [0]:
genre_ratings = df_genres \
    .filter(F.col("total_reviews") >= 100) \
    .groupBy("genre_clean") \
    .agg(
        F.round(F.avg("positive_ratio"), 2).alias("ratio_positif_moyen"),
        F.count("*").alias("nb_jeux")
    ) \
    .filter(F.col("nb_jeux") >= 50) \
    .orderBy(F.col("ratio_positif_moyen").desc())

display(genre_ratings)

genre_clean,ratio_positif_moyen,nb_jeux
Animation & Modeling,81.04,85
Design & Illustration,80.76,92
Casual,79.71,4735
Adventure,79.34,6635
Indie,79.13,10607
Utilities,78.3,155
RPG,77.48,3612
Action,77.03,6989
Simulation,76.07,3708
Strategy,75.89,3676


Databricks visualization. Run in Databricks to view.

### Étape 14 — Q9 : Les top publishers ont-ils des genres préférés ?

On regarde la répartition des genres pour les 5 plus gros publishers.

In [0]:
# Top 5 publishers par nombre de jeux
top5_publishers = df_clean \
    .filter(F.col("publisher").isNotNull() & (F.col("publisher") != "")) \
    .groupBy("publisher") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .limit(5)

# Genres préférés de ces 5 publishers
top5_list = [row.publisher for row in top5_publishers.collect()]

publisher_genres = df_genres \
    .filter(F.col("publisher").isin(top5_list)) \
    .groupBy("publisher", "genre_clean") \
    .count() \
    .orderBy("publisher", F.col("count").desc())

display(publisher_genres)

publisher,genre_clean,count
8floor,Casual,202
8floor,Strategy,22
8floor,Simulation,10
8floor,Adventure,8
8floor,Indie,1
Big Fish Games,Casual,418
Big Fish Games,Adventure,392
Big Fish Games,Indie,7
Big Fish Games,Simulation,7
Big Fish Games,Strategy,6


Databricks visualization. Run in Databricks to view.

### Étape 15 — Q10 : Genres les plus lucratifs

Le "potentiel lucratif" = prix moyen × nombre de jeux du genre. Les jeux gratuits sont exclus pour ne pas fausser le calcul.

In [0]:
lucrative_genres = df_genres \
    .filter(F.col("price") > 0) \
    .groupBy("genre_clean") \
    .agg(
        F.round(F.avg("price"), 2).alias("prix_moyen"),
        F.count("*").alias("nb_jeux"),
        F.round(F.sum("price"), 0).alias("revenu_potentiel")
    ) \
    .orderBy(F.col("revenu_potentiel").desc()) \
    .limit(15)

display(lucrative_genres)

genre_clean,prix_moyen,nb_jeux,revenu_potentiel
Indie,7.5,34765,260630.0
Action,8.92,20582,183588.0
Adventure,9.02,19019,171582.0
Casual,6.38,19418,123836.0
Simulation,10.42,9451,98517.0
Strategy,9.76,9386,91572.0
RPG,10.59,8140,86213.0
Early Access,10.65,5046,53758.0
Sports,10.59,2253,23855.0
Racing,9.41,1882,17716.0


**Observations :**
- Les genres "Indie"ont des prix plus bas mais un volume énorme
- Les genres "Simulation", "Strategy"" ou "RPG" ont le meilleur ratio revenu potentiel/volume de vente.

---
## PARTIE 4 — Analyse par plateformes

Steam supporte 3 OS : Windows, Mac et Linux.

### Étape 16 — Q11 : Sur quelles plateformes les jeux sont-ils disponibles ?

On compte le nombre de jeux disponibles sur chaque OS séparément (un jeu peut être sur plusieurs OS).

In [0]:
platform_dist = df_clean.agg(
    F.sum(F.when(F.col("on_windows"), 1).otherwise(0)).alias("Windows"),
    F.sum(F.when(F.col("on_mac"), 1).otherwise(0)).alias("Mac"),
    F.sum(F.when(F.col("on_linux"), 1).otherwise(0)).alias("Linux")
)

# Pivoter en format long pour Databricks
platform_long = platform_dist.selectExpr(
    "stack(3, 'Windows', Windows, 'Mac', Mac, 'Linux', Linux) as (plateforme, nb_jeux)"
)

display(platform_long)

plateforme,nb_jeux
Windows,55676
Mac,12770
Linux,8458


Databricks visualization. Run in Databricks to view.

**Observations :**
- Il faut sortir obligatoirement le jeu sur Windows. Mac et Linux sont des bonus optionnels selon le coût de portage.

### Étape 17 — Q12 : Certains genres sont-ils préférentiellement sur certaines plateformes ?

On calcule le **taux de présence** de chaque genre sur Mac et Linux (pas la peine de le faire pour Windows qui est ~100% partout).

In [0]:
genre_platforms = df_genres \
    .groupBy("genre_clean") \
    .agg(
        F.count("*").alias("total"),
        F.round(F.sum(F.when(F.col("on_mac"),   1).otherwise(0)) / F.count("*") * 100, 1).alias("pct_mac"),
        F.round(F.sum(F.when(F.col("on_linux"), 1).otherwise(0)) / F.count("*") * 100, 1).alias("pct_linux")
    ) \
    .filter(F.col("total") >= 100) \
    .orderBy(F.col("pct_linux").desc()) \
    .limit(15)

display(genre_platforms)

genre_clean,total,pct_mac,pct_linux
Game Development,159,32.7,22.0
Indie,39681,25.0,17.6
Strategy,10895,27.6,16.8
RPG,9534,23.6,16.0
Adventure,21431,23.5,15.4
Casual,22086,23.2,15.0
Action,23759,19.2,14.2
Racing,2155,19.7,14.1
Simulation,10836,22.5,14.1
Free to Play,3393,24.9,14.0


Databricks visualization. Run in Databricks to view.

## 🏁 PARTIE 5 — Conclusion et recommandations pour Ubisoft

**Bilan de l'analyse :**
- Steam compte plus de 55 000 jeux avec une croissance explosive depuis 2014, avec un marché qui revient à la situation avant Covid.
- Le marché est dominé par Windows.
- L'anglais est la langue universelle, suivie de l'allemand, du français, de l'espagnol, du russe, et du chinois.
- Les genres Simulation, Strategy et RPG sont les plus lucratifs (en termes de rapport CA / volume de vente).
- Il est préférable de cibler un de prix de vente inferieur à 30€, afin de toucher un public plus large sur la plateforme.
- Enfin le marché est de plus en plus saturé, il faut donc essayer de de se diférencier pour que le jeux ressorte au mieu car le cataloge Steam est énorme.